In [ ]:
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from paths import RAW_DIR, PROCESSED_DIR

In [ ]:
orders = pd.read_csv(RAW_DIR / "orders.csv")
prior = pd.read_csv(RAW_DIR / "order_products__prior.csv")
train = pd.read_csv(RAW_DIR / "order_products__train.csv")

In [ ]:
print(train[train['reordered'] == 1].count())

plt.figure(figsize=(8, 4)) # Optional: adjust the figure size
sns.countplot(x='reordered', data=train)
plt.title('Reordered numbers in train csv') # Add a title
plt.show()

In [ ]:
orders.head()

In [ ]:
prior.head()

In [ ]:
train.head()

# Downsample, keep only 2000 users

In [ ]:
# Visualise user 2 data
orders[orders['user_id']==2]

### Choose 2000 random users and filter only sampled users

In [ ]:
sample_users = orders['user_id'].drop_duplicates().sample(n=2000, random_state=42)
orders = orders[orders['user_id'].isin(sample_users)]

prior = prior[prior['order_id'].isin(orders['order_id'])]
train = train[train['order_id'].isin(orders['order_id'])]

### Check columns with same name 

In [ ]:
prior.columns.intersection(orders.columns)

### Attach user_id and order_number to the historical prior dataset

In [ ]:
prior_merged = prior.merge(orders[['order_id', 'user_id', 'order_number', 'days_since_prior_order']], on='order_id')

In [ ]:
# check non null columns
prior_merged.info()

In [ ]:
# Check data with days since prior order is null 
prior_merged[prior_merged['days_since_prior_order'].isna()]

In [ ]:
# Fill first orders having a NaN for days_since_prior_order with 0
prior_merged['days_since_prior_order'] = prior_merged['days_since_prior_order'].fillna(0)

In [ ]:
# Check all the columns are non-null 
prior_merged.info()

In [ ]:
prior_merged.head()

## User Features

In [ ]:
# user_total_orders: Total number of times a user bought an item 
# user_avg_days_between: Average of days between prior and next order (to know the frequency a user buys an item)
user_features = prior_merged.groupby('user_id').agg(
    user_total_orders=('order_number', 'max'),
    user_avg_days_between=('days_since_prior_order', 'mean')
).reset_index()

In [ ]:
user_features

## Product Features

In [ ]:
# prod_total_purchases: Total number of times a product is bought
# prod_reorder_ratio
product_features = prior_merged.groupby('product_id').agg(
    prod_total_purchases=('order_id', 'count'),
    prod_reorder_ratio=('reordered', 'mean')
).reset_index()

In [ ]:
product_features

## User-Product Interaction Features

In [ ]:
# up_total_bought: Number of times a user bought a product
# To find the relationship between user and product to predict if a user will rebuy it 
user_product_features = prior_merged.groupby(['user_id', 'product_id']).agg(
    up_total_bought=('order_id', 'count')
).reset_index()

In [ ]:
user_product_features

# Final dataset

In [ ]:
# Start with the user-product combinations as the base
final_df = user_product_features.copy()

# Attach the user and product features
final_df = final_df.merge(user_features, on='user_id')
final_df = final_df.merge(product_features, on='product_id')

In [ ]:
final_df

In [ ]:
true_reorders = train[train['reordered'] == 1]

# Attach user_id to reorders
train_orders = orders[orders['eval_set'] == 'train'][['order_id', 'user_id']]
train_purchases = true_reorders.merge(train_orders, on='order_id')[['user_id', 'product_id']]
train_purchases = train_purchases[train_purchases['user_id'].isin(final_df['user_id'])]

# Create ML target label after feature engineering
train_purchases['reordered'] = 1

In [ ]:
train_orders

In [ ]:
train_purchases

In [ ]:
final_df = final_df.merge(train_purchases, on=['user_id', 'product_id'], how='left')
final_df['reordered'] = final_df['reordered'].fillna(0).astype(int)
final_df

In [ ]:
final_df = final_df[list(final_df.drop(['reordered'], axis=1).columns) + ['reordered']]

In [ ]:
final_df[final_df['reordered'] == 0].count()

In [ ]:
final_df.info()

# Save to CSV

In [ ]:
FILE_NAME = 'final_features.csv'
try:
    final_df.to_csv(PROCESSED_DIR / FILE_NAME, index=False, mode='x')
except FileExistsError:
    print('File exists!')
    FILE_NAME = 'new_final_features.csv'
    final_df.to_csv(PROCESSED_DIR / FILE_NAME)